# Jonglage - Mustererkennung 2025

Wer sind wir, was machen wir in welchem modul

wofür die jonglage was soll erkannt werden

wie sind wir vorgegangen

verweis auf unser repo mit dem code um die daten aus den videos zu bekommen

## Aufsetzen des Jupiter-Notebooks and laden des DataFrames

In [31]:
import os
import ast
import numpy as np
import pandas as pd
import sklearn.metrics
import matplotlib.pyplot as plt

# Einstellungen für das Anzeigen von Ergebnissen
plt.rcParams["pgf.texsystem"] = "pdflatex"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.size"] = 14.0
plt.rcParams["text.usetex"] = True
plt.rcParams["savefig.bbox"] = "tight"

In [117]:
# DataFrame laden
cwd = os.getcwd()

path_data = cwd + "/test.csv"
df = pd.read_csv(path_data)

# Convertieren der Strings in jeweilige Datenstruktur
for column in df.columns[1:]:
    df[column] = df[column].apply(ast.literal_eval)


In [118]:
# Extrahieren der Klasse aus dem Videonamen
def extract_classifiaction(name: str):
    if name.endswith("0.mp4"):
        return 0
    return 1

df["classification"] = df["video"].apply(extract_classifiaction)

# Kürzen auf die selbe Länge
# Bestimmen der benötigten minimalen Frame anzahl (ab 70 Frames)
# min_frames = min((len(x) for x in df["balls"] if 70 <= len(x) < 10000), default=10000)

min_frames = 1000
for x in df["ball_a"]:
    y = len(x)
    if y < min_frames and y >=70:
        min_frames = y

# def trim_pairs(entry, frame_count):
#     if not isinstance(entry, list):
#         return entry

#     Z = len(entry[0]) - frame_count
#     z = Z // 2
#     if Z % 2 != 0:
#         return [sublist[z + 1:len(sublist) - z] for sublist in entry]
#     else:
#         return [sublist[z:len(sublist) - z] for sublist in entry]

def trim_pairs(entry, frame_count):
    if not isinstance(entry, list):
        return entry

    Z = len(entry) - frame_count
    z = Z // 2
    if Z % 2 != 0:
        return entry[z + 1:len(entry) - z]
    else:
        return entry[z:len(entry) - z]


df = df.applymap(lambda x: trim_pairs(x, min_frames))

mask = df['ball_a'].apply(lambda x: len(x) >= min_frames if isinstance(x, list) else False)

df = df[mask].reset_index(drop=True)

C:\Users\iveev\AppData\Local\Temp\ipykernel_3628\3984260795.py:42: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: trim_pairs(x, min_frames))


In [119]:
def parse_row(row):
    y = row[-1]
    X = row[1] # :-1]
    return X, y

X, y = zip(*df.apply(parse_row, axis=1))


C:\Users\iveev\AppData\Local\Temp\ipykernel_3628\3039550274.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y = row[-1]
C:\Users\iveev\AppData\Local\Temp\ipykernel_3628\3039550274.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  X = row[1] # :-1]


## Klassifikation

- welcher Klassifikator
- warum wie klassifizieren
- nach was klassifizieren

oder so

In [121]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [122]:

gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred = gnb.predict(X_test)
print((X_test.shape[0], (y_test != y_pred).sum()))

ValueError: Found array with dim 3. GaussianNB expected <= 2.